# Kennicutt–Schmidt evolution tracks — cis25 quenched sample

Where do SIMBA's quenched galaxies travel on the **Kennicutt–Schmidt plane**
(log Σ$_{\rm H_2}$ vs log Σ$_{\rm SFR}$) while they quench, and where do they sit compared with the
observed ALMA-C11 quiescent galaxies (z ≈ 0.34–0.43, `obs_data/almac11/ks_table.csv`)?

Sample: the 266 quenched galaxies of `powderday_flux_quenched_m25.ipynb` (10 anchors, z = 0.3–2,
`tables/powderday_quenched_selection.fits`) plus their mass-matched star-forming partners as a
reference cloud at the anchor. Every galaxy is followed through its **critical epochs**
(sSFR peak → SFT quench start → QT quench end → post-quench → H$_2$ trough → anchor) via the
per-anchor histories + progen links, and Σ$_{\rm H_2}$ / Σ$_{\rm SFR}$ are measured from the
**CAESAR member particles (glist / slist)** stored in the reduced particle files
(`build_reduced_particles_job.py`) at each epoch.

Conventions (details in the closing cell):
* **fiducial Σ**: inside the galaxy's own face-on H$_2$ half-mass radius, Σ = 0.5 M / (π R$_{50}^2$)
  — the observed convention (0.5 M$_{\rm H_2}$ / π R$_{\rm CO}^2$, same R for the SFR);
* **fiducial SFR**: archaeological, stars formed in the last 100 Myr (slist); the instantaneous gas
  SFR (glist) and a 25 Myr window are stored too; SFR = 0 → one-particle upper limit;
* fixed apertures r < 1 / 3.16 / 10 kpc (the m25 ladder) as companions (no 0.5 factor);
* simulated H$_2$ ×1.36 (He) for the comparison; Planck15 throughout.

**Run order (cluster, from the repo root, kernel pd39)**
1. Part 0–1 (`ks-00` … `ks-11`): config, critical epochs → `output/cis25/ks_tracks/ks_track_epochs.fits`.
2. Part 2 (`ks-20`, `BUILD_PLAN=True` once): extraction plan → `sbatch` command printed → run it.
3. Part 3 (`ks-30`, `ks-31`): measurements → `ks_track_measurements.fits` + anchor QC.
4. Part 4 (`ks-40` … `ks-60`): join, figures (`output/cis25/plots/ks_tracks/`), summary tables.

Local tests of the measurement code: `python -m pytest tests/test_ks_tracks.py -v`.

In [ ]:
# ── Part 0 — configuration ────────────────────────────────────────────────────
import os
import gc
import glob
import shutil
import tempfile
import warnings
import numpy as np
import h5py
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from astropy.io import fits
from astropy.table import Table, join, vstack
from astropy.cosmology import Planck15 as COSMO      # same as the quenching machinery / histories

from simbanator.io.simba import Simulation
from simbanator.analysis import HDF5BuildHistory
from simbanator.analysis.quenching import find_quenching_times
import ks_tracks_lib as kl                           # repo-root module (pure numpy; tests/test_ks_tracks.py)

# simbanator.analysis sets a huge global font on import — reset to something sane for these figures
plt.rcParams.update({"font.size": 11, "axes.titlesize": 12, "axes.labelsize": 12,
                     "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 8.5})

# ── simulation ────────────────────────────────────────────────────────────────
SIM_NAME = "cis25"        # SIMBA high-res 25 Mpc/h box (m25n512); must exist in ~/.simbanator/config.json
try:
    sim = Simulation(SIM_NAME)
except KeyError as e:
    raise KeyError(
        f"'{SIM_NAME}' is not registered in ~/.simbanator/config.json on this machine.\n"
        "Register it once (adjust paths to where the 25 Mpc snapshots+catalogs live):\n"
        "  from simbanator.io.config import add_simulation\n"
        "  add_simulation('cis25', data_dir='<...>/SIMBA_25',\n"
        "                 catalog_dir='<...>/SIMBA_25/Groups',\n"
        "                 file_format='m25n512_{snap:03d}.hdf5')\n"
        "then add \"snap_z_map\": \"zsnap_map_caesar_box100.txt\" to that entry."
    ) from e
if sim.scale_factors is None:
    raise ValueError(f"'{SIM_NAME}' config has no snap_z_map")
PARTICLE_PREFIX = sim.file_format.split("_{")[0]     # 'm25n512' — reduced-file prefix (job derives the same)

# ── the m25 sample (must match powderday_flux_quenched_m25.ipynb Part 0) ──────
TARGET_REDSHIFTS = [0.3, 0.5, 0.7, 0.85, 1.0, 1.15, 1.3, 1.5, 1.8, 2.0]
MASS_FLOOR, PASSIVE_FACTOR = 10.0, 0.2               # reference only (selection already made)

# ── paths ─────────────────────────────────────────────────────────────────────
OUT      = os.path.join(os.getcwd(), "output", SIM_NAME)
SFHDIR   = os.path.join(OUT, "caesar_sfh")
TABLEDIR = os.path.join(OUT, "tables")
SELECTION_FITS = os.path.join(TABLEDIR, "powderday_quenched_selection.fits")
KSDIR    = os.path.join(OUT, "ks_tracks")
FIGDIR   = os.path.join(OUT, "plots", "ks_tracks")
PLAN_DIR = os.path.join(SFHDIR, "prof_kstracks")
PLAN_PATH = os.path.join(PLAN_DIR, "dust_profile_plan_kstracks.hdf5")
REDUCED_DIR = os.path.join(OUT, "reduced_particles")
REDUCED_PREFIX = PARTICLE_PREFIX
OBS_CSV  = os.path.join(os.getcwd(), "obs_data", "almac11", "ks_table.csv")
EPOCHS_FITS = os.path.join(KSDIR, "ks_track_epochs.fits")
MEAS_FITS   = os.path.join(KSDIR, "ks_track_measurements.fits")
TRACKS_FITS = os.path.join(KSDIR, "ks_tracks.fits")
for _d in (KSDIR, FIGDIR):
    os.makedirs(_d, exist_ok=True)

# ── stages / apertures / floors ───────────────────────────────────────────────
STAGES_KS   = list(kl.STAGES_KS)          # measured: sf_peak, ssfr_min, sft, qt, post_quench, gas_min, anchor
STAGES_PLOT = list(kl.STAGES_PLOT)        # drawn (time order): sf_peak, sft, qt, post_quench, gas_min, anchor
STAGE_LABEL = {"sf_peak": "sSFR peak", "ssfr_min": "sSFR min", "sft": "SFT (quench start)",
               "qt": "QT (quench end)", "post_quench": "post-quench", "gas_min": r"H$_2$ trough",
               "anchor": "anchor"}
STAGE_MARKER = {"sf_peak": "*", "ssfr_min": "x", "sft": "^", "qt": "s", "post_quench": "D",
                "gas_min": "v", "anchor": "o"}
STAGE_COLOR = {"sf_peak": "#1b9e77", "sft": "#e6ab02", "qt": "#d95f02", "post_quench": "#a6761d",
               "gas_min": "#7570b3", "anchor": "#1f1f1f", "ssfr_min": "0.5"}
FIXED_AP_KPC = tuple(kl.FIXED_AP_KPC)     # 1, 3.162, 10 kpc  (labels ap1kpc / ap3kpc / ap10kpc)
AP_LABELS    = list(kl.AP_LABELS)         # + R50_H2 / R50_star / R50_SFR
AP_TITLE = {"ap1kpc": r"$r<1$ kpc", "ap3kpc": r"$r<3.2$ kpc", "ap10kpc": r"$r<10$ kpc",
            "R50_H2": r"$r<R_{50}({\rm H_2})$", "R50_star": r"$r<R_{50}(\star)$",
            "R50_SFR": r"$r<R_{50}({\rm SFR})$"}
FIDUCIAL_AP  = "R50_H2"                   # observed convention: 0.5 M / (pi R_CO^2)
FIDUCIAL_SFR = "sfr100"                   # archaeological 100 Myr window (slist)
MEMBER_ONLY  = True                       # CAESAR glist / slist particles only
NGAS_MIN, NH2_MIN, NSTAR_MIN = 10, 5, 10  # particle floors (aperture gas / R50_H2 / R50_star)
SFR_WINDOWS_MYR = (25.0, 100.0)
HE_FACTOR = kl.HE_FACTOR                  # 1.36: simulated H2 is hydrogen-only, observed alpha_CO includes He
INCLUDE_SF_CONTROL = True                 # SF partners at their anchor as a reference cloud
Z_OBS = 0.37                              # ALMA-C11 median redshift (Tacconi+18 MS t_dep evaluated here)
A_TO_T = kl.make_a_to_t(COSMO)            # star formation scale factor -> cosmic time [Gyr]

# ── gates (heavy or one-off steps; caches are reused when the flags are False) ──
BUILD_PLAN             = False            # Part 2: write the SLURM extraction plan
OVERWRITE_EPOCHS       = False            # Part 1 cache
OVERWRITE_MEASUREMENTS = False            # Part 3 cache


def _ztag(z):
    return ("z%g" % z).replace(".", "p")


def _s(col):
    """FITS string column -> stripped str array (bytes on some numpy/astropy combos)."""
    return np.char.strip(np.asarray(col).astype(str))


def write_table(tab, path):
    """astropy Table -> FITS via a temp file in the same directory (atomic; gvfs-safe pattern)."""
    d = os.path.dirname(path)
    os.makedirs(d, exist_ok=True)
    fd, tmp = tempfile.mkstemp(prefix=".tmp_", suffix=".fits", dir=d)
    os.close(fd)
    tab.write(tmp, overwrite=True)
    os.replace(tmp, path)
    return path


print(f"sim={sim.name}  prefix={PARTICLE_PREFIX}  anchors z={TARGET_REDSHIFTS}")
print(f"fiducial: aperture={FIDUCIAL_AP}  SFR={FIDUCIAL_SFR}  member_only={MEMBER_ONLY}  He x{HE_FACTOR}")
print("stages:", STAGES_KS)
print("selection:", SELECTION_FITS, "->", "ok" if os.path.exists(SELECTION_FITS) else "MISSING")
print("observed:", OBS_CSV, "->", "ok" if os.path.exists(OBS_CSV) else "MISSING (git pull?)")

In [ ]:
# ── anchor table (snapshot nearest each target z; per-anchor history / progenitor products) ──
_sall, _zall = [], []
for _s_ in range(0, 152):
    try:
        _zv = float(sim.get_z_from_snap(_s_))
    except Exception:
        continue
    if np.isfinite(_zv) and _zv >= 0:
        _sall.append(_s_); _zall.append(_zv)
_sall, _zall = np.asarray(_sall), np.asarray(_zall)

ANCHORS = {}
for _zt in TARGET_REDSHIFTS:
    _snap = int(_sall[np.argmin(np.abs(_zall - _zt))])
    _tag = _ztag(_zt)
    ANCHORS[_zt] = dict(z_target=_zt, tag=_tag, snap=_snap, z=float(sim.get_z_from_snap(_snap)),
                        prog_file=f"progenitors_anchor_{_tag}.fits",
                        hist_path=os.path.join(SFHDIR, f"history_anchor_{_tag}.hdf5"))

print(f"{'z_tgt':>6s} {'snap':>5s} {'z':>7s} {'hist':>6s} {'prog':>6s}")
for _zt, A in ANCHORS.items():
    _pf = os.path.join(OUT, "progenitors", A["prog_file"])
    print(f"{_zt:6.2f} {A['snap']:5d} {A['z']:7.3f} "
          f"{'ok' if os.path.exists(A['hist_path']) else '--':>6s} "
          f"{'ok' if os.path.exists(_pf) else '--':>6s}")

In [ ]:
# ── loaders (verbatim from powderday_flux_quenched_m25 Parts 0b/1): row 0 = the anchor epoch ──
def load_selection():
    """SELECTION_FITS (written by the m25 notebook Part 3) -> (table, snap array, gal_id array)."""
    sel = Table.read(SELECTION_FITS)
    return sel, np.asarray(sel["snap"], int), np.asarray(sel["gal_id"], int)


def load_anchor_history(A):
    """Load one anchor's history -> dict(galaxy_ids, snaps_arr, redshift, t_cosmic_yr, P)."""
    H = {"P": {}}
    with h5py.File(A["hist_path"], "r") as f:
        H["galaxy_ids"] = f["metadata/galaxy_ids"][:]
        H["snaps_arr"]  = f["metadata/snapshots"][:]
        H["redshift"]   = f["redshift/Redshift"][:]
        f["properties"].visititems(
            lambda name, obj: H["P"].__setitem__(name, obj[:]) if isinstance(obj, h5py.Dataset) else None)
    H["t_cosmic_yr"] = COSMO.age(H["redshift"]).value * 1e9
    return H


def build_prog_index(A, galaxy_ids, snaps_arr):
    """(n_snap, n_gal) catalogue group-index matrix aligned to the anchor history rows
    (walks tree_data/progen_galaxy_star through the progen_links sidecars; cwd must be the repo root)."""
    cs0 = sim.load_catalog(snap=A["snap"])
    hP = HDF5BuildHistory(sim, cs0, progfilename=A["prog_file"])
    hP.get_history_indx(galaxy_ids, int(np.max(snaps_arr)), int(np.min(snaps_arr)))
    M = np.vstack([hP.history_indx[str(s)] for s in snaps_arr])
    del cs0, hP
    gc.collect()
    return M

## Part 1 — critical epochs of every quenched galaxy

One row per (anchor, galaxy, stage) with the snapshot + catalogue index of the progenitor at that
epoch (`gx`, −1 when the progenitor chain is broken or the stage lies beyond the anchor), the
epoch times, and the catalogue-level history values at that row (for the Part 3 cross-check).
`t_qt` is recomputed with the same finder and the same history as the selection table and must
agree exactly. Star-forming partners (`pop = SF`) get an `anchor` row only (they have no history).

In [ ]:
# ── Part 1 — epochs table (cached) ──
_HIST_KEYS = {"mstar_cat": "masses.stellar", "sfr_cat": "sfr", "mh2_cat": "masses.H2",
              "mgas_cat": "masses.gas", "mdust_cat": "masses.dust",
              "r50star_cat": "radii.stellar_half_mass", "r50gas_cat": "radii.gas_half_mass",
              "ngas_cat": "ngas", "nstar_cat": "nstar"}

EPOCHS = None
if os.path.exists(EPOCHS_FITS) and not OVERWRITE_EPOCHS:
    EPOCHS = Table.read(EPOCHS_FITS)
    print(f"cached ({len(EPOCHS)} rows) -> {EPOCHS_FITS}   (OVERWRITE_EPOCHS=True rebuilds)")
if EPOCHS is None:
    SEL, SNAPS, IDS = load_selection()
    POP = _s(SEL["pop"])
    _rows, _tqt_check = [], []
    for _zt, A in ANCHORS.items():
        _inA = SNAPS == A["snap"]
        _q = SEL[_inA & (POP == "Q")]
        _sf = SEL[_inA & (POP == "SF")]
        if len(_q) == 0 and len(_sf) == 0:
            continue
        print(f"[{A['tag']}] snap {A['snap']} z={A['z']:.2f}: {len(_q)} Q, {len(_sf)} SF")
        if len(_q):
            H = load_anchor_history(A)
            _g2c = {int(g): j for j, g in enumerate(H["galaxy_ids"])}
            _missing = [int(g) for g in _q["gal_id"] if int(g) not in _g2c]
            if _missing:
                raise KeyError(f"[{A['tag']}] {len(_missing)} selected galaxies absent from the history: {_missing[:10]}")
            _cols = np.array([_g2c[int(g)] for g in _q["gal_id"]], int)
            _recs = kl.build_stage_records(H["P"], H["t_cosmic_yr"], H["redshift"], H["galaxy_ids"],
                                           _cols, find_quenching_times,
                                           age_of_z_gyr=lambda z: COSMO.age(z).value)
            PIDX = build_prog_index(A, H["galaxy_ids"], H["snaps_arr"])
            for _rec, _qrow in zip(_recs, _q):
                _tqt_check.append((float(_qrow["t_qt"]), _rec["t_qt"]))
                for _st in STAGES_KS:
                    _row = int(_rec["row_%s" % _st])
                    _tdef = _rec["t_%s" % _st]
                    if _row < 0 and not np.isfinite(_tdef):
                        continue                      # stage undefined for this galaxy
                    _gx = PIDX[_row, _rec["col"]] if _row >= 0 else np.nan
                    _t = float(H["t_cosmic_yr"][_row]) / 1e9 if _row >= 0 else np.nan
                    _d = dict(anchor_z=float(_zt), anchor_snap=int(A["snap"]), gal_id=int(_rec["gid"]),
                              pop="Q", stage=_st, row=_row,
                              snap=int(H["snaps_arr"][_row]) if _row >= 0 else -1,
                              gx=int(_gx) if np.isfinite(_gx) else -1,
                              z=float(H["redshift"][_row]) if _row >= 0 else np.nan,
                              t_stage_gyr=_t, t_def_gyr=_tdef / 1e9 if np.isfinite(_tdef) else np.nan,
                              dt_from_qt_gyr=(_t - _rec["t_qt"] / 1e9) if (np.isfinite(_t) and np.isfinite(_rec["t_qt"])) else np.nan,
                              t_sft_gyr=_rec["t_sft"] / 1e9, t_qt_gyr=_rec["t_qt"] / 1e9,
                              tau_q_gyr=_rec["tau_q"] / 1e9, z_qt=_rec["z_qt"], n_events=int(_rec["n_events"]),
                              agn_class=str(_s(np.array([_qrow["agn_class"]]))[0]),
                              xstr_quench=float(_qrow["xstr_quench"]),
                              log_mstar_anchor=float(_qrow["log_mstar"]))
                    for _k, _pk in _HIST_KEYS.items():
                        _d[_k] = float(H["P"][_pk][_row, _rec["col"]]) if (_row >= 0 and _pk in H["P"]) else np.nan
                    _rows.append(_d)
            del H, PIDX
            gc.collect()
        if INCLUDE_SF_CONTROL and len(_sf):
            _tA = float(COSMO.age(A["z"]).value)
            for _r in _sf:
                _ms = 10.0 ** float(_r["log_mstar"])
                _d = dict(anchor_z=float(_zt), anchor_snap=int(A["snap"]), gal_id=int(_r["gal_id"]),
                          pop="SF", stage="anchor", row=0, snap=int(A["snap"]), gx=int(_r["gal_id"]),
                          z=float(A["z"]), t_stage_gyr=_tA, t_def_gyr=_tA, dt_from_qt_gyr=np.nan,
                          t_sft_gyr=np.nan, t_qt_gyr=np.nan, tau_q_gyr=np.nan, z_qt=np.nan, n_events=0,
                          agn_class="SF", xstr_quench=np.nan, log_mstar_anchor=float(_r["log_mstar"]),
                          mstar_cat=_ms, sfr_cat=float(_r["ssfr"]) * _ms, mh2_cat=float(_r["mh2"]),
                          mgas_cat=np.nan, mdust_cat=float(_r["mdust"]), r50star_cat=np.nan,
                          r50gas_cat=np.nan, ngas_cat=float(_r["ngas"]), nstar_cat=float(_r["nstar"]))
                _rows.append(_d)
    EPOCHS = Table(rows=_rows)
    # the selection's t_qt came from the same finder on the same history: must agree exactly
    _c = np.array(_tqt_check, float)
    _both = np.isfinite(_c).all(axis=1)
    _dmax = np.abs(_c[_both, 0] - _c[_both, 1]).max() if _both.any() else 0.0
    _nfin = int((np.isfinite(_c[:, 0]) != np.isfinite(_c[:, 1])).sum())
    print(f"t_qt check vs selection: {int(_both.sum())} finite pairs, max |dt| = {_dmax:.3g} yr, "
          f"{_nfin} finiteness mismatches")
    if _dmax > 1e3 or _nfin:
        warnings.warn("recomputed t_qt disagrees with powderday_quenched_selection.fits — "
                      "history files or find_quenching_times changed since the selection was made")
    write_table(EPOCHS, EPOCHS_FITS)
    print(f"wrote {len(EPOCHS)} rows -> {EPOCHS_FITS}")

_st_col, _pop_col = _s(EPOCHS["stage"]), _s(EPOCHS["pop"])
print("rows per stage (Q):", {st: int(((_st_col == st) & (_pop_col == "Q")).sum()) for st in STAGES_KS})
print("SF control rows:", int((_pop_col == "SF").sum()),
      "| Q stage rows with a tracked progenitor (gx>=0):",
      int(((_pop_col == "Q") & (np.asarray(EPOCHS["gx"]) >= 0)).sum()))

In [ ]:
# ── Part 1 QC — stage availability funnel, coincident epochs, time-since-QT per stage ──
_stg, _pop = _s(EPOCHS["stage"]), _s(EPOCHS["pop"])
_gx = np.asarray(EPOCHS["gx"], int)
_isQ = _pop == "Q"
print(f"{'anchor':>7s} " + " ".join(f"{st[:9]:>11s}" for st in STAGES_KS) + "   (defined/tracked)")
for _zt, A in ANCHORS.items():
    _inA = _isQ & (np.asarray(EPOCHS["anchor_snap"]) == A["snap"])
    if not _inA.any():
        continue
    _cells = []
    for st in STAGES_KS:
        _m = _inA & (_stg == st)
        _cells.append(f"{int(_m.sum()):4d}/{int((_m & (_gx >= 0)).sum()):<4d}")
    print(f"{_zt:7.2f} " + " ".join(f"{c:>11s}" for c in _cells))

# coincident epochs: several stages of one galaxy on the same (snap, gx) -> one reduced file serves them
_keyQ = [(int(a), int(g), int(s), int(x)) for a, g, s, x in
         zip(EPOCHS["anchor_snap"][_isQ & (_gx >= 0)], EPOCHS["gal_id"][_isQ & (_gx >= 0)],
             EPOCHS["snap"][_isQ & (_gx >= 0)], _gx[_isQ & (_gx >= 0)])]
_uniq = len(set(_keyQ))
print(f"\nQ epoch rows with a file: {len(_keyQ)} -> {_uniq} distinct (galaxy, snap) -> "
      f"{len(set((s, x) for _, _, s, x in _keyQ))} distinct reduced files (snap, gx)")
_beyond = _isQ & (_stg == "post_quench") & (_gx < 0) & np.isfinite(np.asarray(EPOCHS["t_def_gyr"]))
print(f"post_quench beyond the anchor (not observable in the history): {int(_beyond.sum())} galaxies")

fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
for st in ("sft", "qt", "post_quench", "gas_min", "anchor"):
    _m = _isQ & (_stg == st) & (_gx >= 0) & np.isfinite(np.asarray(EPOCHS["dt_from_qt_gyr"]))
    if _m.sum():
        axs[0].hist(np.asarray(EPOCHS["dt_from_qt_gyr"])[_m], bins=np.linspace(-4, 8, 49),
                    histtype="step", lw=1.6, color=STAGE_COLOR[st], label=f"{STAGE_LABEL[st]} (N={int(_m.sum())})")
axs[0].axvline(0, color="0.5", lw=0.8)
axs[0].set_xlabel(r"$t_{\rm stage} - t_{\rm QT}$ [Gyr]"); axs[0].set_ylabel("galaxies"); axs[0].legend()
axs[0].set_title("(a) time of each stage relative to QT", loc="left")
_m = _isQ & (_stg == "anchor")
axs[1].scatter(np.asarray(EPOCHS["z"])[_m], np.asarray(EPOCHS["z_qt"])[_m], s=12, c="0.3")
axs[1].plot([0, 3], [0, 3], color="0.6", lw=0.8)
axs[1].set_xlabel("anchor z"); axs[1].set_ylabel(r"$z_{\rm QT}$"); axs[1].set_title("(b) when they quenched", loc="left")
plt.tight_layout(); fig.savefig(os.path.join(FIGDIR, "ks_epochs_qc.png"), dpi=150); plt.show()

## Part 2 — extraction plan for the reduced particle files

The unique (snapshot, catalogue index) pairs of Part 1 go into ONE plan file in the schema
`build_reduced_particles_job.py` reads (`sim_name` attr + `entry_gx` / `entry_snap`), plus a
provenance group the job ignores. The job is idempotent: files that already exist (e.g. the
box-comparison campaign at snaps 105/134) are only **backfilled** with the new `tform` field
(star formation epochs → archaeological SFR); nothing is re-extracted.
Set `BUILD_PLAN=True`, run this cell once, then submit on the cluster (command printed below).

In [ ]:
# ── Part 2 — plan (gated) ──
_ok = (np.asarray(EPOCHS["gx"], int) >= 0) & (np.asarray(EPOCHS["snap"], int) > 0)
_pairs = sorted(set((int(s), int(g)) for s, g in zip(EPOCHS["snap"][_ok], EPOCHS["gx"][_ok])))
_snaps_u = sorted(set(s for s, _ in _pairs))
_have = [os.path.exists(kl.reduced_path(REDUCED_DIR, REDUCED_PREFIX, s, g)) for s, g in _pairs]
print(f"{len(_pairs)} unique (snap, gx) over {len(_snaps_u)} snapshots "
      f"({min(_snaps_u)}..{max(_snaps_u)}); reduced files already on disk: {sum(_have)}")

SBATCH_CMD = ("cd /mnt/home/glorenzon/analize_simba_cgm && mkdir -p logs && "
              f"DUST_PLAN={os.path.relpath(PLAN_PATH, os.getcwd())} "
              "sbatch --array=0-15%4 submit_reduced_particles.sh")
if BUILD_PLAN:
    os.makedirs(PLAN_DIR, exist_ok=True)
    _tmp = PLAN_PATH + ".part"
    with h5py.File(_tmp, "w") as f:
        f.attrs["sim_name"] = SIM_NAME
        f.attrs["note"] = "ks_tracks_quenched_m25.ipynb: quenched sample at its critical epochs (+ SF controls at the anchor)"
        f.attrs["rmax_kpc"] = 100.0
        f.attrs["stages"] = ",".join(STAGES_KS)
        f.attrs["selection_fits"] = SELECTION_FITS
        f.attrs["n_epochs"] = int(_ok.sum())
        f.create_dataset("entry_gx", data=np.array([g for _, g in _pairs], np.int64))
        f.create_dataset("entry_snap", data=np.array([s for s, _ in _pairs], np.int32))
        m = f.create_group("map")                        # provenance, ignored by the job
        _E = EPOCHS[_ok]
        for c in ("anchor_snap", "gal_id", "snap", "gx"):
            m.create_dataset(c, data=np.asarray(_E[c], np.int64))
        for c in ("pop", "stage"):
            m.create_dataset(c, data=np.asarray(_s(_E[c]), dtype=object), dtype=h5py.string_dtype())
    os.replace(_tmp, PLAN_PATH)
    print(f"wrote plan -> {PLAN_PATH}")
    print("submit on the cluster (16 array tasks over snapshots, throttled to 4 concurrent):\n  " + SBATCH_CMD)
elif os.path.exists(PLAN_PATH):
    with h5py.File(PLAN_PATH, "r") as f:
        _pp = set(zip(f["entry_snap"][:].astype(int), f["entry_gx"][:].astype(int)))
    _new = set(_pairs) - _pp
    print(f"existing plan: {len(_pp)} entries; {len(_new)} current pairs NOT in it "
          + ("(BUILD_PLAN=True to refresh)" if _new else "(up to date)"))
    print("submit / resubmit with:\n  " + SBATCH_CMD)
else:
    print("no plan yet -> set BUILD_PLAN=True and re-run this cell")

### Cluster steps between Part 2 and Part 3

```bash
cd /mnt/home/glorenzon/analize_simba_cgm && mkdir -p logs
DUST_PLAN=output/cis25/caesar_sfh/prof_kstracks/dust_profile_plan_kstracks.hdf5 \
  sbatch --array=0-15%4 submit_reduced_particles.sh
```
* 16 array tasks split the plan's snapshots (newest first); `%4` keeps at most four 32 GB tasks
  running at once — SLURM here schedules on cores only, so un-throttled memory-hungry arrays
  OOM-kill each other on the shared phi nodes.
* Progress: `grep -h "snap" logs/reduced_*.out` (per snapshot: N planned → extracted / backfilled /
  already complete); `grep -h done: logs/reduced_*.out` when finished. Re-submitting the same command
  fills any gaps (idempotent).
* Then run Part 3. Files still missing are reported there (job not finished / progenitor without
  a catalogue entry).

## Part 3 — Σ$_{\rm H_2}$ and SFR from the member particles at every epoch

Each reduced file → face-on cylindrical radii in the stored stellar principal frame
(`pos @ evecs`, columns = axes) → CAESAR members only → in every aperture (three fixed rungs and the
three half-mass radii): H$_2$/HI/gas/dust mass, instantaneous gas SFR, archaeological SFR over
25 / 100 Myr from the star formation epochs, particle counts. One row per (snap, gx, aperture);
member totals and the half-mass radii are repeated on every row of a file.

In [ ]:
# ── Part 3 — measurements (cached) ──
MEAS = None
if os.path.exists(MEAS_FITS) and not OVERWRITE_MEASUREMENTS:
    MEAS = Table.read(MEAS_FITS)
    print(f"cached ({len(MEAS)} rows, {len(set(zip(MEAS['snap'], MEAS['gx'])))} files) -> {MEAS_FITS}   "
          "(OVERWRITE_MEASUREMENTS=True rebuilds)")
if MEAS is None:
    _rows, _bad = [], []
    _n_missing = _n_notform = 0
    _KEYS = {"gas": ("pos", "m_gas", "m_H2", "m_HI", "m_dust", "sfr", "member"),
             "star": ("pos", "m_star", "member", "tform")}
    for _k, (_snap, _gx) in enumerate(_pairs):
        _p = kl.reduced_path(REDUCED_DIR, REDUCED_PREFIX, _snap, _gx)
        _red = kl.load_reduced(_p, keys=_KEYS, bad=_bad)
        if _red is None:
            _n_missing += 1
            continue
        _zs = float(_red["redshift"])
        _t_obs = float(COSMO.age(_zs).value)
        _rr = kl.measure_ks(_red, _t_obs, A_TO_T, fixed_kpc=FIXED_AP_KPC, member_only=MEMBER_ONLY,
                            ngas_min=NGAS_MIN, nh2_min=NH2_MIN, nstar_min=NSTAR_MIN,
                            sfr_windows=SFR_WINDOWS_MYR)
        if not _rr[0]["has_tform"]:
            _n_notform += 1
        for _r in _rr:
            _r.update(snap=int(_snap), gx=int(_gx), z_file=_zs, t_obs_gyr=_t_obs)
            _rows.append(_r)
        if (_k + 1) % 200 == 0:
            print(f"  {_k + 1}/{len(_pairs)} files")
    MEAS = Table(rows=_rows)
    write_table(MEAS, MEAS_FITS)
    print(f"measured {len(_pairs) - _n_missing - len(_bad)}/{len(_pairs)} files -> {MEAS_FITS}")
    print(f"  missing files: {_n_missing} (job not finished?)   corrupt: {len(_bad)}   "
          f"without tform (old files not yet backfilled): {_n_notform}")

_apl = _s(MEAS["ap_label"])
for _lab in AP_LABELS:
    _m = _apl == _lab
    _fin = _m & np.isfinite(np.asarray(MEAS["m_H2"], float)) & (np.asarray(MEAS["m_H2"], float) > 0)
    print(f"  {_lab:9s}: {int(_m.sum())} rows, Sigma_H2 measurable (n_gas>={NGAS_MIN}, M_H2>0): {int(_fin.sum())}, "
          f"SFR100=0: {int((_m & (np.asarray(MEAS['sfr100'], float) == 0)).sum())}")

In [ ]:
# ── Part 3 QC — member totals at the anchor vs the catalogue (progenitor index + member flag + units) ──
_E = EPOCHS[(_s(EPOCHS["stage"]) == "anchor") & (np.asarray(EPOCHS["gx"]) >= 0)]
_M = MEAS[_s(MEAS["ap_label"]) == "R50_H2"]           # any label carries the file totals
_J = join(_E, _M, keys=("snap", "gx"), join_type="inner")
_popJ = _s(_J["pop"])
print(f"anchor rows joined: {len(_J)} ({int((_popJ == 'Q').sum())} Q, {int((_popJ == 'SF').sum())} SF)")


def _ratio_stats(name, num, den):
    num, den = np.asarray(num, float), np.asarray(den, float)
    ok = np.isfinite(num) & np.isfinite(den) & (den > 0) & (num > 0)
    if not ok.any():
        print(f"  {name:28s}: no finite pairs"); return None
    r = np.log10(num[ok] / den[ok])
    print(f"  {name:28s}: N={int(ok.sum()):4d}  median log(sim/cat) = {np.median(r):+.3f}  "
          f"16-84 = [{np.percentile(r, 16):+.3f}, {np.percentile(r, 84):+.3f}]")
    return r


print("member totals vs catalogue (expect ~0 for masses; radii differ by definition: face-on 2D vs 3D):")
_rm = _ratio_stats("M_H2 (member sum / cat)", _J["m_H2_tot"], _J["mh2_cat"])
_rs = _ratio_stats("M* (member sum / cat)", _J["m_star_tot"], _J["mstar_cat"])
_rf = _ratio_stats("SFR_inst (member / cat)", _J["sfr_inst_tot"], _J["sfr_cat"])
_ra = _ratio_stats("SFR100 (archaeol. / cat inst)", _J["sfr100_tot"], _J["sfr_cat"])
_rr = _ratio_stats("R50* face-on / cat half-mass", _J["r50_star"], _J["r50star_cat"])

fig, axs = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, (xk, yk, lab) in zip(axs, (("mh2_cat", "m_H2_tot", r"$M_{\rm H_2}$ [M$_\odot$]"),
                                   ("mstar_cat", "m_star_tot", r"$M_\star$ [M$_\odot$]"),
                                   ("sfr_cat", "sfr_inst_tot", r"SFR$_{\rm inst}$ [M$_\odot$ yr$^{-1}$]"))):
    x, y = np.asarray(_J[xk], float), np.asarray(_J[yk], float)
    ok = (x > 0) & (y > 0)
    ax.scatter(x[ok & (_popJ == "SF")], y[ok & (_popJ == "SF")], s=8, c="0.6", label="SF control")
    ax.scatter(x[ok & (_popJ == "Q")], y[ok & (_popJ == "Q")], s=10, c="#d95f02", label="Q")
    lo, hi = np.nanmin(np.r_[x[ok], y[ok]]), np.nanmax(np.r_[x[ok], y[ok]])
    ax.plot([lo, hi], [lo, hi], color="0.4", lw=0.8)
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("catalogue " + lab); ax.set_ylabel("member sum " + lab)
axs[0].legend(); axs[0].set_title("anchor epoch: member particles reproduce the catalogue?", loc="left")
plt.tight_layout(); fig.savefig(os.path.join(FIGDIR, "ks_anchor_qc.png"), dpi=150); plt.show()

## Part 4 — the Kennicutt–Schmidt plane

Join epochs × apertures, scale to the observational conventions (`ks_tracks_lib.ks_columns`), overlay
the observed ALMA-C11 galaxies and the published relations, and draw the tracks.

In [ ]:
# ── Part 4a — join + KS columns (fiducial = archaeological 100 Myr; also 25 Myr and instantaneous) ──
_E = EPOCHS[np.asarray(EPOCHS["gx"]) >= 0]
TRACKS = join(_E, MEAS, keys=("snap", "gx"), join_type="inner")      # epoch rows x 6 aperture rows
for _k, _v in kl.ks_columns(TRACKS, "sfr100", "sfr100_tot", 100.0, HE_FACTOR).items():
    TRACKS[_k] = _v
for _suf, _key, _tot, _w in (("_sfr25", "sfr25", "sfr25_tot", 25.0), ("_inst", "sfr_inst", "sfr_inst_tot", 100.0)):
    for _k, _v in kl.ks_columns(TRACKS, _key, _tot, _w, HE_FACTOR, suffix=_suf).items():
        if _k != "logSigmaH2":
            TRACKS[_k] = _v
TRACKS["tdep_ms_gyr"] = kl.tdep_ms_gyr(np.asarray(TRACKS["z"], float))
TRACKS["gkey"] = np.array([f"{a}_{g}" for a, g in zip(TRACKS["anchor_snap"], TRACKS["gal_id"])])
write_table(TRACKS, TRACKS_FITS)
print(f"TRACKS: {len(TRACKS)} rows -> {TRACKS_FITS}")

# node export in the observed table's column names (for pilot_specphot/scripts/plot_ks.py overlays)
_ap, _stg, _pop = _s(TRACKS["ap_label"]), _s(TRACKS["stage"]), _s(TRACKS["pop"])
_nodes = pd.DataFrame(dict(
    id=np.asarray(TRACKS["gkey"]).astype(str), pop=_pop, stage=_stg, aperture=_ap,
    z=np.asarray(TRACKS["z"], float), anchor_z=np.asarray(TRACKS["anchor_z"], float),
    dt_from_qt_gyr=np.asarray(TRACKS["dt_from_qt_gyr"], float),
    agn_class=_s(TRACKS["agn_class"]), log_mstar=np.log10(np.asarray(TRACKS["mstar_cat"], float)),
    r_kpc=np.asarray(TRACKS["ap_kpc"], float),
    MH2_fid=HE_FACTOR * np.asarray(TRACKS["m_H2_tot"], float), SFR=np.asarray(TRACKS["sfr100_tot"], float),
    logSigmaH2=np.asarray(TRACKS["logSigmaH2"], float), logSigmaSFR=np.asarray(TRACKS["logSigmaSFR"], float),
    is_ul=np.asarray(TRACKS["is_ul"], bool), tdep_Gyr=np.asarray(TRACKS["tdep_gyr"], float),
    logSigmaSFR_inst=np.asarray(TRACKS["logSigmaSFR_inst"], float), n_gas=np.asarray(TRACKS["n_gas"], int),
    n_H2=np.asarray(TRACKS["n_H2"], int)))
_nodes.to_csv(os.path.join(KSDIR, "ks_track_nodes.csv"), index=False)
print(f"nodes CSV -> {os.path.join(KSDIR, 'ks_track_nodes.csv')}")
_fid = (_ap == FIDUCIAL_AP) & (_pop == "Q")
print(f"fiducial ({FIDUCIAL_AP}, {FIDUCIAL_SFR}) Q rows: {int(_fid.sum())}; both Sigma finite: "
      f"{int((_fid & np.isfinite(TRACKS['logSigmaH2']) & np.isfinite(TRACKS['logSigmaSFR_obs'])).sum())}; "
      f"SFR censored (upper limits): {int((_fid & np.asarray(TRACKS['is_ul'], bool)).sum())}; "
      f"R50_H2 undefined: {int((_fid & ~np.isfinite(np.asarray(TRACKS['ap_kpc'], float))).sum())}")

In [ ]:
# ── Part 4b — observed ALMA-C11 points, reference relations, plotting helpers ──
OBS = pd.read_csv(OBS_CSV)
OBS_S = OBS[OBS["co_det"].astype(bool) & np.isfinite(OBS["logSigmaH2"])].reset_index(drop=True)
print(f"observed: {len(OBS)} sources, {len(OBS_S)} with a surface density "
      f"({int(OBS_S['sigma_is_ll'].astype(bool).sum())} Sigma lower limits); z = {OBS['z'].min():.2f}-{OBS['z'].max():.2f}")
C_OBS, C_OBS_EDGE = "#e7298a", "#3b0f2a"
XLIM, YLIM = (-0.5, 3.6), (-4.6, 0.8)


def draw_relations(ax, xs=None, tdep_lines=(0.1, 1.0, 10.0)):
    """K98 / B08 / RK19 with their published scatter bands + constant-t_dep lines. Returns legend items."""
    xs = np.linspace(XLIM[0] - 0.5, XLIM[1] + 0.5, 80) if xs is None else xs
    items = []
    for t, ls in zip(tdep_lines, (":", "-", "--")):
        ax.plot(xs, xs + 6 - np.log10(t * 1e9), color="0.65", ls=ls, lw=0.9, zorder=1)
        ax.text(XLIM[1] - 0.05, XLIM[1] - 0.05 + 6 - np.log10(t * 1e9) - 0.25, rf"$t_{{\rm dep}}$={t:g} Gyr",
                fontsize=7.5, color="0.45", ha="right", rotation=42, rotation_mode="anchor")
    for key, rel in kl.RELATIONS.items():
        y = rel["A"] + rel["N"] * xs
        ax.fill_between(xs, y - rel["sig"], y + rel["sig"], color=rel["color"], alpha=0.10, lw=0, zorder=0)
        ax.plot(xs, y, color=rel["color"], lw=1.4, ls=rel["ls"], zorder=2)
        items.append((Line2D([], [], color=rel["color"], lw=1.4, ls=rel["ls"]),
                      rf"{rel['label']} $\pm$ {rel['scat_label']}"))
    return items


def overlay_obs(ax, annotate=True):
    """ALMA-C11 points: filled = size measured (asymmetric errors), open + slope-1 arrow = unresolved
    (both Sigma lower limits). Returns legend items."""
    for _, r in OBS_S.iterrows():
        x, y = float(r["logSigmaH2"]), float(r["logSigmaSFR"])
        if bool(r["sigma_is_ll"]):
            ax.annotate("", xy=(x + 0.3, y + 0.3), xytext=(x, y),
                        arrowprops=dict(arrowstyle="-|>", color=C_OBS_EDGE, lw=1.2, mutation_scale=10), zorder=8)
            ax.scatter(x, y, s=110, marker="o", facecolors="white", edgecolors=C_OBS_EDGE, linewidths=1.4, zorder=9)
            ax.scatter(x, y, s=40, marker="o", c=C_OBS, edgecolors="none", zorder=10)
        else:
            elo = r["logSigmaSFR_elo"] if np.isfinite(r["logSigmaSFR_elo"]) else 0.0
            ehi = r["logSigmaSFR_ehi"] if np.isfinite(r["logSigmaSFR_ehi"]) else 0.0
            elo = min(elo, y - YLIM[0])                     # unbounded lower errors run to the axis floor
            xe = r["logSigmaH2_err"] if np.isfinite(r["logSigmaH2_err"]) else 0.0
            ax.errorbar(x, y, xerr=xe, yerr=[[elo], [ehi]], fmt="none", ecolor=C_OBS_EDGE, elinewidth=1.0, zorder=8)
            ax.scatter(x, y, s=110, marker="o", c=C_OBS, edgecolors=C_OBS_EDGE, linewidths=1.4, zorder=9)
        if annotate:
            ax.annotate(str(r["id"]), (x, y), xytext=(5, 4), textcoords="offset points", fontsize=7, color="0.25", zorder=11)
    return [(Line2D([], [], marker="o", ls="", ms=9, mfc=C_OBS, mec=C_OBS_EDGE),
             rf"ALMA-C11 QGs, $z\approx0.4$ (N={len(OBS_S)}; size measured)"),
            (Line2D([], [], marker="o", ls="", ms=9, mfc="white", mec=C_OBS_EDGE), r"ALMA-C11 unresolved: $\Sigma$ lower limits")]


def ks_axes(ax, ap_label, sfr_note="SFR over 100 Myr (stars)"):
    conv = r"$0.5M/\pi R_{50}^2$" if ap_label.startswith("R50") else r"$M(<r)/\pi r^2$"
    ax.set_xlabel(r"$\log(\Sigma_{\rm H_2}/M_\odot\,{\rm pc}^{-2})$  [" + AP_TITLE[ap_label] + ", " + conv + r", $\times1.36$ He]")
    ax.set_ylabel(r"$\log(\Sigma_{\rm SFR}/M_\odot\,{\rm yr}^{-1}\,{\rm kpc}^{-2})$  [" + sfr_note + "]")
    ax.set_xlim(XLIM); ax.set_ylim(YLIM)
    ax.tick_params(direction="in", top=True, right=True); ax.grid(alpha=0.15, lw=0.6)


def _sel_ap(ap_label, pop="Q"):
    return TRACKS[(_s(TRACKS["ap_label"]) == ap_label) & (_s(TRACKS["pop"]) == pop)]


def stage_medians(T, xcol="logSigmaH2", ycol="logSigmaSFR_obs", stages=STAGES_PLOT, nmin=5):
    """Per-stage median (16-84) of x and y over rows with both finite and the SFR not censored."""
    stg, ul = _s(T["stage"]), np.asarray(T["is_ul"], bool)
    x, y = np.asarray(T[xcol], float), np.asarray(T[ycol], float)
    out = []
    for st in stages:
        m = (stg == st) & np.isfinite(x) & np.isfinite(y) & ~ul
        n_ul = int(((stg == st) & np.isfinite(x) & ul).sum())
        if m.sum() < nmin:
            out.append(dict(stage=st, n=int(m.sum()), n_ul=n_ul)); continue
        out.append(dict(stage=st, n=int(m.sum()), n_ul=n_ul,
                        x=np.median(x[m]), x16=np.percentile(x[m], 16), x84=np.percentile(x[m], 84),
                        y=np.median(y[m]), y16=np.percentile(y[m], 16), y84=np.percentile(y[m], 84)))
    return out

In [ ]:
# ── Figure 1 — KS tracks: fiducial (R50_H2, 100 Myr SFR) + the fixed 3.2 / 10 kpc rungs ──
from matplotlib import colormaps
from matplotlib.colors import Normalize
Z_NORM = Normalize(vmin=min(TARGET_REDSHIFTS), vmax=max(TARGET_REDSHIFTS))
Z_CMAP = colormaps["viridis"]


def track_panel(ax, ap_label, ycol="logSigmaSFR", show_tracks=True, show_sf=True, show_obs=True,
                annotate_obs=False, max_tracks=None, medians=True, title=None):
    """One KS panel: per-galaxy tracks through the stages (colour = anchor z), stage medians with
    16-84 % bars, SF-control cloud, censored SFR as down arrows, observed points, relations."""
    T = _sel_ap(ap_label, "Q")
    stg, ul = _s(T["stage"]), np.asarray(T["is_ul"], bool)
    x, y = np.asarray(T["logSigmaH2"], float), np.asarray(T[ycol], float)
    az, gk = np.asarray(T["anchor_z"], float), np.asarray(T["gkey"]).astype(str)
    tt = np.asarray(T["t_stage_gyr"], float)
    n_gal = 0
    if show_tracks:
        for k, g in enumerate(np.unique(gk)):
            if max_tracks is not None and k >= max_tracks:
                break
            m = (gk == g) & np.isfinite(x) & np.isfinite(y) & np.isin(stg, STAGES_PLOT)
            if m.sum() < 2:
                continue
            o = np.argsort(tt[m])
            col = Z_CMAP(Z_NORM(az[m][0]))
            ax.plot(x[m][o], y[m][o], "-", color=col, lw=0.7, alpha=0.45, zorder=3)
            for xi, yi, si, ui in zip(x[m][o], y[m][o], stg[m][o], ul[m][o]):
                ax.scatter(xi, yi, s=14 if si != "anchor" else 18, marker=STAGE_MARKER[si],
                           c=[col], edgecolors="none", alpha=0.7, zorder=4)
                if ui:
                    ax.annotate("", xy=(xi, yi - 0.25), xytext=(xi, yi),
                                arrowprops=dict(arrowstyle="-|>", color=col, lw=0.6, alpha=0.6, mutation_scale=6), zorder=4)
            n_gal += 1
    if show_sf:
        S = _sel_ap(ap_label, "SF")
        xs_, ys_ = np.asarray(S["logSigmaH2"], float), np.asarray(S[ycol], float)
        ax.scatter(xs_, ys_, s=7, c="0.55", alpha=0.35, lw=0, zorder=2)
    meds = stage_medians(T, ycol=ycol) if medians else []
    for md_ in meds:
        if "x" not in md_:
            continue
        ax.errorbar(md_["x"], md_["y"], xerr=[[md_["x"] - md_["x16"]], [md_["x84"] - md_["x"]]],
                    yerr=[[md_["y"] - md_["y16"]], [md_["y84"] - md_["y"]]], fmt="none", ecolor="k",
                    elinewidth=1.0, capsize=2, alpha=0.8, zorder=6)
        ax.scatter(md_["x"], md_["y"], s=140, marker=STAGE_MARKER[md_["stage"]], c=[STAGE_COLOR[md_["stage"]]],
                   edgecolors="k", linewidths=1.0, zorder=7)
    mx = [m for m in meds if "x" in m]
    if len(mx) >= 2:
        ax.plot([m["x"] for m in mx], [m["y"] for m in mx], "-", color="k", lw=2.0, alpha=0.8, zorder=6)
    rel_items = draw_relations(ax)
    obs_items = overlay_obs(ax, annotate=annotate_obs) if show_obs else []
    ks_axes(ax, ap_label, "SFR over 100 Myr (stars)" if ycol.startswith("logSigmaSFR") and "inst" not in ycol else "instantaneous gas SFR")
    if title:
        ax.set_title(title, loc="left")
    return n_gal, meds, rel_items, obs_items


fig, axs = plt.subplots(1, 3, figsize=(19, 6.2))
_panels = [(FIDUCIAL_AP, "(a) fiducial: inside the H$_2$ half-mass radius"),
           ("ap3kpc", "(b) fixed $r<3.2$ kpc"), ("ap10kpc", "(c) fixed $r<10$ kpc")]
for ax, (ap, ttl) in zip(axs, _panels):
    n_gal, meds, rel_items, obs_items = track_panel(ax, ap, title=ttl, annotate_obs=(ap == FIDUCIAL_AP))
    print(f"{ap}: {n_gal} galaxy tracks;", "  ".join(
        f"{m['stage']}: N={m['n']} (+{m['n_ul']} UL)" + (f" x={m['x']:.2f} y={m['y']:.2f}" if 'x' in m else "") for m in meds))
# legends: stages (medians), anchor colour bar, relations + observed
_stage_items = [(Line2D([], [], marker=STAGE_MARKER[st], ls="", ms=9, mfc=STAGE_COLOR[st], mec="k"), STAGE_LABEL[st])
                for st in STAGES_PLOT]
_misc = [(Line2D([], [], color="k", lw=2.0), "stage medians (16–84 %); thin lines = individual galaxies"),
         (Line2D([], [], marker="o", ls="", ms=5, mfc="0.55", mec="none"), "SF controls at their anchor"),
         (Line2D([], [], marker="v", ls="", ms=6, mfc="none", mec="0.3"), "SFR = 0: one-particle upper limit")]
axs[0].legend([h for h, _ in _stage_items + _misc], [l for _, l in _stage_items + _misc], loc="upper left", frameon=False)
axs[1].legend([h for h, _ in rel_items + obs_items], [l for _, l in rel_items + obs_items], loc="upper left", frameon=False)
_sm = plt.cm.ScalarMappable(cmap=Z_CMAP, norm=Z_NORM); _sm.set_array([])
fig.colorbar(_sm, ax=axs, fraction=0.02, pad=0.01, label="anchor redshift of the track")
fig.suptitle("SIMBA-25 quenched galaxies on the Kennicutt–Schmidt plane through their critical epochs "
             "(member particles; Σ_SFR = 100 Myr archaeological)", y=0.995)
fig.savefig(os.path.join(FIGDIR, "ks_tracks_plane.png"), dpi=160, bbox_inches="tight")
fig.savefig(os.path.join(FIGDIR, "ks_tracks_plane.pdf"), bbox_inches="tight")
plt.show()

# companion: the same fiducial aperture with the INSTANTANEOUS gas SFR (SIMBA's SF law directly)
fig, ax = plt.subplots(figsize=(7.2, 6.2))
track_panel(ax, FIDUCIAL_AP, ycol="logSigmaSFR_inst", title="fiducial aperture, instantaneous gas SFR (glist)")
fig.savefig(os.path.join(FIGDIR, "ks_tracks_plane_inst.png"), dpi=160, bbox_inches="tight"); plt.show()

In [ ]:
# ── Figure 2 — depletion clock: t_dep(H2) vs time since QT, by AGN coupling class ──
T = _sel_ap(FIDUCIAL_AP, "Q")
_dt = np.asarray(T["dt_from_qt_gyr"], float)
_td = np.asarray(T["tdep_gyr"], float)
_stg, _cls = _s(T["stage"]), _s(T["agn_class"])
_ok = np.isfinite(_dt) & np.isfinite(_td) & (_td > 0)
CLASSES = [("weak", "#1b9e77"), ("intermediate", "#7570b3"), ("strong", "#d95f02")]
_obs_td = OBS["tdep_Gyr"][OBS["co_det"].astype(bool)].values
_obs_td = _obs_td[np.isfinite(_obs_td)]

fig, axs = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw=dict(width_ratios=[3, 1.2]))
ax = axs[0]
for st in STAGES_PLOT:
    m = _ok & (_stg == st)
    ax.scatter(_dt[m], np.log10(_td[m]), s=16, marker=STAGE_MARKER[st], c=[STAGE_COLOR[st]], alpha=0.55,
               label=f"{STAGE_LABEL[st]} (N={int(m.sum())})", zorder=3)
_bins = np.array([-4, -2, -1, -0.5, -0.25, 0, 0.25, 0.75, 1.25, 2, 3, 5, 8])
for cname, ccol in CLASSES:
    m = _ok & (_cls == cname)
    xs_, ys_ = [], []
    for lo, hi in zip(_bins[:-1], _bins[1:]):
        mb = m & (_dt >= lo) & (_dt < hi)
        if mb.sum() >= 4:
            xs_.append(np.median(_dt[mb])); ys_.append(np.median(np.log10(_td[mb])))
    if xs_:
        ax.plot(xs_, ys_, "-", color=ccol, lw=2.2, zorder=5, label=f"median, {cname} coupling (N={int(m.sum())})")
ax.axvline(0, color="0.5", lw=0.8)
if _obs_td.size:
    ax.axhspan(np.log10(np.percentile(_obs_td, 16)), np.log10(np.percentile(_obs_td, 84)), color=C_OBS, alpha=0.15, lw=0,
               label=rf"ALMA-C11 $t_{{\rm dep}}$ 16–84 % (N={len(_obs_td)})")
ax.axhline(np.log10(kl.tdep_ms_gyr(Z_OBS)), color="0.3", ls="--", lw=1.0, label=f"Tacconi+18 MS at z={Z_OBS}")
ax.set_xlabel(r"$t - t_{\rm QT}$ [Gyr]"); ax.set_ylabel(r"$\log(t_{\rm dep}/{\rm Gyr})$  [" + AP_TITLE[FIDUCIAL_AP] + ", 100 Myr SFR]")
ax.set_title("(a) molecular depletion time along the quench clock", loc="left"); ax.legend(ncol=2, fontsize=8)
ax = axs[1]
_anc = _ok & (_stg == "anchor")
ax.hist(np.log10(_td[_anc]), bins=np.linspace(-1.5, 2.5, 25), color="0.6", label=f"SIMBA anchors (N={int(_anc.sum())})")
if _obs_td.size:
    ax.hist(np.log10(_obs_td), bins=np.linspace(-1.5, 2.5, 25), histtype="step", lw=2, color=C_OBS, label="ALMA-C11")
ax.set_xlabel(r"$\log(t_{\rm dep}/{\rm Gyr})$"); ax.set_ylabel("N"); ax.legend(fontsize=8)
ax.set_title("(b) at the anchor vs observed", loc="left")
plt.tight_layout(); fig.savefig(os.path.join(FIGDIR, "ks_tracks_tdep_clock.png"), dpi=160); plt.show()

In [ ]:
# ── Figure 3 — the fiducial plane in three anchor-redshift bins (z <= 0.5 is the ALMA-C11-matched set) ──
Z_BINS = [(0.0, 0.55, r"anchors $z\leq0.5$ (observed z ≈ 0.34–0.43)"), (0.55, 1.05, r"anchors $0.7\leq z\leq1$"),
          (1.05, 2.5, r"anchors $1.15\leq z\leq2$")]
fig, axs = plt.subplots(1, 3, figsize=(19, 6.2))
for ax, (zlo, zhi, ttl) in zip(axs, Z_BINS):
    T_all = TRACKS
    _mz = (np.asarray(T_all["anchor_z"], float) > zlo) & (np.asarray(T_all["anchor_z"], float) <= zhi)
    TRACKS_BAK = TRACKS
    TRACKS = T_all[_mz]                                  # track_panel reads the global
    try:
        n_gal, meds, _, _ = track_panel(ax, FIDUCIAL_AP, title=ttl, show_obs=(zlo == 0.0), annotate_obs=False)
    finally:
        TRACKS = TRACKS_BAK
    print(f"{ttl}: {n_gal} tracks; anchor median:",
          next((f"x={m['x']:.2f} y={m['y']:.2f} (N={m['n']}, UL {m['n_ul']})" for m in meds if m['stage'] == 'anchor' and 'x' in m), "n/a"))
fig.savefig(os.path.join(FIGDIR, "ks_tracks_plane_by_anchor.png"), dpi=160, bbox_inches="tight"); plt.show()

In [ ]:
# ── Part 4 summary — per (stage, aperture): N, censoring, medians, offsets from the relations ──
_rows = []
for ap in AP_LABELS:
    T = _sel_ap(ap, "Q")
    stg, ul = _s(T["stage"]), np.asarray(T["is_ul"], bool)
    x, y = np.asarray(T["logSigmaH2"], float), np.asarray(T["logSigmaSFR_obs"], float)
    td, rk = np.asarray(T["tdep_gyr"], float), np.asarray(T["ap_kpc"], float)
    for st in STAGES_PLOT:
        m_all = stg == st
        m = m_all & np.isfinite(x) & np.isfinite(y) & ~ul
        r = dict(aperture=ap, stage=st, n_rows=int(m_all.sum()), n_sigma_h2=int((m_all & np.isfinite(x)).sum()),
                 n_both=int(m.sum()), n_sfr_ul=int((m_all & np.isfinite(x) & ul).sum()),
                 f_r50_undefined=float(np.mean(~np.isfinite(rk[m_all]))) if m_all.any() else np.nan)
        for key, arr in (("logSigmaH2", x), ("logSigmaSFR", y), ("log_tdep_gyr", np.log10(np.where(td > 0, td, np.nan))),
                         ("r_kpc", rk)):
            v = arr[m]
            v = v[np.isfinite(v)]
            r[key + "_med"] = float(np.median(v)) if v.size else np.nan
            r[key + "_p16"] = float(np.percentile(v, 16)) if v.size else np.nan
            r[key + "_p84"] = float(np.percentile(v, 84)) if v.size else np.nan
        for key in kl.RELATIONS:
            d = y[m] - kl.relation_y(key, x[m])
            r["dlog_" + key + "_med"] = float(np.median(d)) if d.size else np.nan
        _rows.append(r)
SUMMARY = Table(rows=_rows)
write_table(SUMMARY, os.path.join(KSDIR, "ks_stage_summary.fits"))
SUMMARY.to_pandas().to_csv(os.path.join(KSDIR, "ks_stage_summary.csv"), index=False)
_show = SUMMARY[_s(SUMMARY["aperture"]) == FIDUCIAL_AP]
print(f"fiducial aperture {FIDUCIAL_AP} (Q galaxies; medians over rows with both Sigma finite and SFR>0):")
for r in _show:
    print(f"  {r['stage']:12s} N={r['n_both']:3d} (+{r['n_sfr_ul']:3d} SFR UL, R50 undef {100*r['f_r50_undefined']:4.0f}%)  "
          f"logSigH2={r['logSigmaH2_med']:+.2f} [{r['logSigmaH2_p16']:+.2f},{r['logSigmaH2_p84']:+.2f}]  "
          f"logSigSFR={r['logSigmaSFR_med']:+.2f}  log tdep={r['log_tdep_gyr_med']:+.2f}  "
          f"R50={r['r_kpc_med']:.2f} kpc  dlog B08={r['dlog_B08_med']:+.2f} K98={r['dlog_K98_med']:+.2f}")
print("\nobserved ALMA-C11 (CO-detected with size):")
print(OBS_S[["id", "z", "logSigmaH2", "logSigmaSFR", "sigma_is_ll", "r_kpc", "tdep_Gyr", "dlog_B08", "dlog_K98"]]
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))

## Conventions, caveats, and what the pieces are

* **Particles**: only CAESAR members (`gal.glist` / `gal.slist`, the `member` flag of the reduced
  files); the 100 kpc aperture of the files (CGM, satellites) is ignored. Face-on = the stellar
  principal frame stored in each file (`pos @ evecs`, columns = axes — the older
  `quench_mode_vs_sigma_gas` / `box_resolution_comparison` helper uses `evecs.T`, which is the wrong
  frame). For spheroidal quenched galaxies the frame is nearly degenerate and "face-on" is only nominal.
* **Σ$_{\rm H_2}$**: SIMBA's per-particle H$_2$ (`m_H2 = m_gas X_H f_neut f_mol`) is hydrogen-only; the
  observed M$_{\rm H_2}$ (α$_{\rm CO}$ = 4.36) includes helium → ×1.36 on the simulated values in the
  plots and summaries; raw masses stay in the tables.
* **0.5 factor**: inside a half-mass radius M(<R$_{50}$)/πR$_{50}^2$ ≡ 0.5 M$_{\rm tot}$/πR$_{50}^2$, so the
  fiducial rows reproduce the observed convention with no extra factor. For the SFR the observed
  points assume the SFR follows the CO (0.5 SFR$_{\rm tot}$/πR$_{\rm CO}^2$): that is `logSigmaSFR_obs`
  (used on the R50_H2 rows); `logSigmaSFR_inside` is the literal SFR(<R)/area. Fixed apertures never
  carry a 0.5 — they are comparable to the `cluster_tracks.py` overlay of the pilot_specphot project.
* **SFR**: fiducial = mass of member stars formed within 100 Myr of the snapshot / 100 Myr (current
  masses, no mass-loss correction, ~5–10 %); `sfr25` and the instantaneous gas SFR (`sfr_inst`, exactly
  0 when no gas sits above the SF density threshold) are stored. SFR = 0 → censored: the one-particle
  floor m$_\star$/100 Myr/area is drawn as a downward arrow and excluded from the medians.
* **Stages**: `sf_peak` = peak of the (median-3 smoothed) sSFR history; `sft`/`qt` = the last quench
  event of `find_quenching_times` (1/t and 0.2/t crossings with persistence), identical to the selection
  table; `post_quench` = end of the persistence window (QT + 0.2 QT) — often beyond the anchor, then
  not measured; `gas_min` = trough of the H$_2$ mass after QT (before any floor); `anchor` = the
  selection snapshot. Galaxies without a detected event (28/266) have sf_peak / gas_min / anchor only.
* **Floors**: Σ from gas needs ≥ `NGAS_MIN` = 10 gas particles in the aperture, R$_{50}$(H$_2$) needs
  ≥ `NH2_MIN` = 5 H$_2$-bearing particles (else the fiducial row is NaN — the summary reports the
  fraction); cis25 particle masses are ~1.6×10$^6$ M$_\odot$ (stars) so the 100 Myr SFR floor is
  ~0.016 M$_\odot$ yr$^{-1}$.
* **Cosmology**: Planck15 throughout (histories, quench finder, formation times) — SIMBA's own
  h = 0.68, Ω$_m$ = 0.30 differ at the sub-percent level in ages.
* **Observed side**: `obs_data/almac11/ks_table.csv` (see its README) — α$_{\rm CO}$ 4.36, R$_{31}$ 0.5,
  Σ = 0.5 M/πR$^2$ with R = CO(3–2) uv FWHM$_{\rm maj}$/2; unresolved sizes → Σ lower limits (slope-1
  arrows). Reference relations: Kennicutt 98 (Chabrier-scaled), Bigiel+08 (H$_2$, 750 pc), de los
  Reyes & Kennicutt 19, with their published scatter; Tacconi+18 main-sequence t$_{\rm dep}$ at z = 0.37.
* **Products** (`output/cis25/ks_tracks/`): `ks_track_epochs.fits` (Part 1), `ks_track_measurements.fits`
  (Part 3), `ks_tracks.fits` + `ks_track_nodes.csv` (Part 4; the CSV uses the observed table's column
  names so `pilot_specphot/scripts/plot_ks.py` can overlay it), `ks_stage_summary.fits/.csv`.
  Figures in `output/cis25/plots/ks_tracks/`.